# Batch image analysis with LLMFlux

Run a vision-language model over a folder of images on a Slurm GPU cluster, and get one structured JSON record back per image.

The prompt is the main thing you edit: it says which fields you want (here a summary, the objects, the scene, any visible text and a few flags), and the model fills them in for every image.

**Steps**
1. Configure: where your images are, which model, which Slurm account.
2. Write the prompt.
3. Build the batch requests (one per image) with `llmflux.converters.vision_to_jsonl`.
4. Submit the batch job with `llmflux run`.
5. Check on the job.
6. Parse the replies into a table and a JSON file.

**You need**
- A Jupyter kernel whose Python can `import llmflux`. On NCSA Delta, `module load llmflux` provides one (the module's conda environment includes Jupyter).
- A Slurm account with GPU access (`accounts` lists yours on Delta).
- Images. The folder `sample_images/` next to this notebook has four small synthetic ones, so it runs as is.

Steps 1–3 and 6 run on the login node or wherever Jupyter runs; only step 4 uses a GPU.

*Based on an original notebook by Ajay Nagi Reddy Kumdam.*

## 1) Configure

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
from llmflux.converters import vision_to_jsonl
from llmflux.converters.vision import IMAGE_MIME_TYPES

NOTEBOOK_DIR = Path.cwd()

# -------- Input images --------
IMAGE_DIR = Path(os.environ.get("LLMFLUX_IMAGE_DIR", NOTEBOOK_DIR / "sample_images"))
MAX_IMAGES = 50  # None to use every image in IMAGE_DIR

# -------- Where this run's files go --------
WORKDIR = Path(os.environ.get("LLMFLUX_WORKDIR", NOTEBOOK_DIR / "vision_run"))
REQUESTS_JSONL = WORKDIR / "requests.jsonl"   # the batch input
RESULTS_JSON = WORKDIR / "results.json"       # written by the batch job
PARSED_JSON = WORKDIR / "parsed.json"         # written by step 6

# -------- Model and generation --------
MODEL = "Qwen2.5-VL-7B-Instruct"  # a vision model key from `llmflux show-models`
TEMPERATURE = 0.1                 # low: we want consistent, structured answers
MAX_TOKENS = 400

# -------- Slurm --------
ACCOUNT = os.environ.get("SLURM_ACCOUNT", "")  # required for step 4, e.g. <project>-delta-gpu
PARTITION = os.environ.get("SLURM_PARTITION", "gpuA100x4")
TIME_LIMIT = "00:15:00"  # one A100: ~2 min for 50 images; ~4.5 min the first time (model download)
BATCH_SIZE = 1

# vLLM settings: 8192 tokens of context fits a prompt plus one image comfortably.
VLLM_ENGINE_ARGS = {
    "max-model-len": 8192,
    "gpu-memory-utilization": 0.95,
    "max-num-seqs": 1,
}

WORKDIR.mkdir(parents=True, exist_ok=True)
print("Images from:", IMAGE_DIR)
print("Run files in:", WORKDIR)

## 2) Write the prompt

This is the main customization point. Change the JSON structure to the fields you need; the parsing in step 6 doesn't assume any particular keys.

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise image analysis assistant. "
    "Follow the output format exactly. Do not add extra prose outside JSON."
)

USER_PROMPT = """\
Analyze this image and return ONLY a valid JSON object with this structure:
{
  "summary": string,
  "objects": [string],
  "scene": string,
  "visible_text": [string],
  "safety_flags": {
    "violence": boolean,
    "adult": boolean,
    "medical": boolean
  }
}

Rules:
- Return JSON only (no markdown, no explanation).
- If a field is unknown, use null or an empty list as appropriate.
- Keep "summary" to one or two sentences.
"""

## 3) Build the batch requests

`vision_to_jsonl` writes one request per image, with the image embedded (base64) next to your prompt, in the OpenAI batch format `llmflux run` reads.

Two adjustments afterwards:
- The converter writes `temperature: 0.7` and `max_tokens: 500` into every request, and a request's own values take priority over `llmflux run --temperature/--max-tokens`. So the values from step 1 are set on each request here.
- The `model` field is removed: `llmflux run --model` fills in the engine's own model name.

In [ ]:
input_dir = IMAGE_DIR
if MAX_IMAGES is not None:
    # Link the first MAX_IMAGES images into a subset folder, so a large folder
    # isn't encoded in full just to keep a few of its images.
    images = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in IMAGE_MIME_TYPES)[:MAX_IMAGES]
    input_dir = WORKDIR / "input_subset"
    input_dir.mkdir(exist_ok=True)
    for old in input_dir.iterdir():
        if old.is_symlink():
            old.unlink()
    for image in images:
        (input_dir / image.name).symlink_to(image.resolve())

_, report = vision_to_jsonl(
    str(input_dir),
    str(REQUESTS_JSONL),
    prompt_template=USER_PROMPT,
    system_prompt=SYSTEM_PROMPT,
    return_report=True,
)

requests = [json.loads(line) for line in REQUESTS_JSONL.read_text().splitlines() if line.strip()]
for request in requests:
    request["body"].pop("model", None)
    request["body"]["temperature"] = TEMPERATURE
    request["body"]["max_tokens"] = MAX_TOKENS
REQUESTS_JSONL.write_text("".join(json.dumps(r) + "\n" for r in requests))

print(f"Converted {report['converted']} of {report['found']} images into {REQUESTS_JSONL}")
for skipped in report["skipped_too_large"]:
    print("  skipped (too large):", skipped["path"])
for failed in report["failed"]:
    print("  failed:", failed["path"], failed["error"])

A look at the first request, with the image data shortened:

In [ ]:
first = json.loads(json.dumps(requests[0]))
image_part = first["body"]["messages"][-1]["content"][1]["image_url"]
image_part["url"] = image_part["url"][:60] + "..."
print(json.dumps(first, indent=2)[:1500])

## 4) Submit the batch job

This is the only step that uses a GPU. `llmflux run` writes a Slurm job script, submits it, and returns right away; the job loads the model, answers every request, and writes `RESULTS_JSON`.

The first run downloads the model's weights (about 16 GB for Qwen2.5-VL-7B) into `HF_HOME`, which defaults to a cache inside your LLMFlux workspace. On a cluster, point `HF_HOME` at project or scratch space before starting Jupyter rather than letting it land in your home directory, and reuse that cache for later runs.

LLMFlux is called through this kernel's own Python (`python -m llmflux.cli`), so it works even if the `llmflux` command isn't on the kernel's PATH.

In [ ]:
def llmflux(*args):
    """Run an llmflux CLI command with this kernel's Python and print its output."""
    result = subprocess.run([sys.executable, "-m", "llmflux.cli", *args], capture_output=True, text=True)
    print(result.stdout + result.stderr)
    return result

assert ACCOUNT, "Set SLURM_ACCOUNT (or ACCOUNT in step 1) to your GPU account. `accounts` lists yours on Delta."

submitted = llmflux(
    "run",
    "--engine", "vllm",
    "--model", MODEL,
    "--input", str(REQUESTS_JSONL),
    "--output", str(RESULTS_JSON),
    "--account", ACCOUNT,
    "--partition", PARTITION,
    "--gpus-per-node", "1",
    "--time", TIME_LIMIT,
    "--batch-size", str(BATCH_SIZE),
    "--vllm-engine-args", json.dumps(VLLM_ENGINE_ARGS),
)
match = re.search(r"Job ID:\s*(\d+)", submitted.stdout)
JOB_ID = match.group(1) if match else None
print("Job ID:", JOB_ID)

## 5) Check on the job

Re-run this cell until the job shows COMPLETED. While it's PENDING it's waiting for a GPU; while RUNNING, the log shows the model loading and then the requests being processed.

In [ ]:
llmflux("status", JOB_ID)
llmflux("logs", JOB_ID, "--tail", "30")

## 6) Parse the replies

Each reply should be a JSON object, but models sometimes wrap it in a ```` ```json ```` fence or add a sentence, so the parser takes the outermost `{...}`. Every image ends up with one of three statuses:
- **ok**: the reply parsed as a JSON object
- **reply not JSON**: the model answered, but not in the requested format
- **request failed**: the request itself failed (see its `error`)

Check a few "ok" rows by hand, too: valid JSON can still be wrong about what's in the image.

In [ ]:
def parse_json_reply(text):
    """The JSON object in a model reply, or None if there isn't one."""
    if not isinstance(text, str):
        return None
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        value = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None


data = json.loads(RESULTS_JSON.read_text())
results = data["results"] if isinstance(data, dict) else data  # older LLMFlux wrote a bare list

rows = []
for item in results:
    row = {"image": item["input"]["custom_id"]}
    if item.get("error") or not item.get("output"):
        row.update(status="request failed", error=item.get("error"))
    else:
        reply = item["output"]["choices"][0]["message"]["content"]
        parsed = parse_json_reply(reply)
        row.update(status="ok" if parsed is not None else "reply not JSON", reply=reply, parsed=parsed)
    rows.append(row)

PARSED_JSON.write_text(json.dumps(rows, indent=2, ensure_ascii=False))
print(pd.Series([r["status"] for r in rows]).value_counts().to_string())
print("Saved", PARSED_JSON)

table = pd.DataFrame([
    {
        "image": r["image"],
        "status": r["status"],
        "summary": (r.get("parsed") or {}).get("summary"),
        "objects": ", ".join((r.get("parsed") or {}).get("objects") or []),
        "visible_text": ", ".join((r.get("parsed") or {}).get("visible_text") or []),
    }
    for r in rows
])
table

## Next steps

- **Your own images:** set `LLMFLUX_IMAGE_DIR` (or `IMAGE_DIR` in step 1). Supported types: JPEG, PNG, GIF, WebP, BMP. Images over 25 MB are skipped and listed in step 3.
- **More images:** set `MAX_IMAGES = None` and raise `TIME_LIMIT`. Every image is embedded in the request file, so a large folder makes a large file; splitting it into several runs keeps each job short.
- **Different fields:** edit `USER_PROMPT`; step 6 works with any JSON structure.
- **A different model:** `llmflux show-models` lists the vision models LLMFlux knows (look for `VL`, `vision`, `llava`, `InternVL`, `Pixtral`).